# Opening and processing actigraphy data

Welcome to `circStudio`! This introduction provides an overview of how to load actigraphy data using the `Raw` class and demonstrates how adaptor subclasses can be used to convert common actigraphy file formats into `Raw` instances.

We begin the tutorial by importing `circStudio`, `Numpy`, `Pandas` and `os`:

In [ ]:
import circStudio
import numpy as np
import pandas as pd
import os

## Reading actigraphy files using the `Raw` class directly

The preferred method for loading actigraphy data in `circStudio` is to create a new instance of the `Raw` class. This requires the actigraphy data to be preformatted as a table containing an activity and light series.

<div class="alert alert-block alert-info"> 
<b>NOTE</b> Unlike <code>pyActigraphy</code>, <code>circStudio</code> decouples the computation of actigraphy metrics from the <code>Raw</code> class, which is used exclusively for preprocessing the actigraphy signal, allowing users greater flexibility. For instance, users may define custom preprocessing pipelines or apply <code>circStudio</code> metrics to other types of time series data, such as skin temperature.
</div>

Unlike `pyActigraphy`, `circStudio` decouples the computation of actigraphy metrics from the `Raw` class, which is used exclusively for preprocessing the actigraphy signal, allowing users greater flexibility. For instance, users may define custom preprocessing pipelines or apply `circStudio` metrics to other types of time series data, such as skin temperature.

To illustrate the creation of a `Raw` object, we first create a new `pd.DataFrame` filled with random data sampled from a normal distribution. First, we initialize a default random number generator (RNG):

In [ ]:
rng = np.random.default_rng()

Next, we generate synthetic actigraphy data by creating random time series for activity and light. These values are indexed using a `DatetimeIndex` to simulate a seven-day recording period with a sampling frequency of sixty seconds. This step is intended for demonstration purposes and can be skipped if the user already has actigraphy data stored in a `pandas.DataFrame`:

In [ ]:
# Define the number of samples for seven days of data at a 60s interval
n = 1440*7 # 1440 minutes/day x 7 days

# Generate a random time series to simulate activity counts
activity = rng.normal(loc=10,scale=1, size=n)

# Generate a random time series to simulate light exposure (in lux)
light = rng.normal(loc=100, scale=10, size=n)

# Create a datetime index starting on January 1st, 2025, with 60s intervals
index = pd.date_range(start='01-01-2025', freq='60s', periods=n)

# Store the synthetic data into a pd.DataFrame
data = pd.DataFrame(index=index,
                    data={
                        'Activity': activity,
                        'Light': light
                    })

# Display the resulting DataFrame
data

Finally, we import the `Raw`class from `circStudio.io` and create a new `Raw` instance using the synthetic data. We specify the dataframe (`df`), activity (`activity`) and light (`light`) time serie, start time (`start_time`), total duration (`period`), and sampling frequency (`frequency`):

In [ ]:
# Import the Raw class from circStudio.io
from circStudio.io import Raw

# Create a new Raw instance
raw = Raw(
    df=data, # pd.DataFrame
    activity=data['Activity'], # Activity time series
    light=data['Light'], # Light time series
    start_time=data.index[0], # Start time
    period=(data.index[-1]-data.index[0]), # Total duration
    frequency=data.index.freq # Sampling frequency
)

## Reading actigraphy files using adaptor subclasses

To simplify the data import process, `circStudio` includes several adaptor subclasses for commonly used actigraphy file formats. These adaptors enable users to easily convert supported file types into `Raw` instances.

<div class="alert alert-block alert-info"> 
<b>NOTE</b> The <code>Raw</code> class is designed to be format-agnostic and flexible. Users can implement custom functions to convert data from other actigraphy file formats for which <code>circStudio</code> does not natively have an adaptor, as long as the resulting data is compatible with the structure expected by the <code>Raw</code> class. As observed in the previous section, a <code>Raw</code> object requires the user to provide a DataFrame containing all the data (<code>df</code>), activity (<code>activity</code>) and light (<code>light</code>) time series, start time (<code>start_time</code>), total duration (<code>period</code>`), and sampling frequency (<code>frequency</code>). 
</div>

In the following example, we open a `.txt` file generated by an ActTrust (Condor Instruments) actigraphy device. To access example files included with `circStudio`, we construct a file path using `os.path`:

In [ ]:
fpath = os.path.join(os.path.dirname(circStudio.__file__))

This retrieves the directory where `circStudio` is installed by referencing its `__file__` attribute, which stores the path from which `circStudio` was imported. `os.path.dirname` extracts the directory name from that path.

Next, we create a new `Raw` instance using the auxiliary function `read_atr`, which is located in `circStudio.io`:

In [ ]:
raw = circStudio.io.read_atr(os.path.join(fpath, 'data', 'test_sample_atr.txt'))

<div class="alert alert-block alert-warning"> 
<b>WARNING</b> Some ATR files contain an extra line above the header, such as <code>#ActLogModel=2.0.0</code>, which should should be excluded when importing data. The <code>skip_rows</code> optional parameter allows users to skip lines during import:
    <code>circStudio.io.read_atr(os.path.join(fpath, 'data', 'test_sample_atr.txt'), skip_rows=1)</code>
</div>

Each adaptor subclass can be accessed using a helper function with the format `read_XXX`, where `XXX` indicates the file format. Besides `read_atr`, the available adaptor functions are:

In [ ]:
# Actiwatch
raw = circStudio.io.read_awd(os.path.join(fpath, 'data', 'example_01.AWD'))

In [ ]:
# Actigraph
raw = circStudio.io.read_agd(os.path.join(fpath, 'data', 'test_sample.agd'))

In [ ]:
# Daqtometer
raw = circStudio.io.read_dqt(os.path.join(fpath, 'data', 'test_sample_dqt.csv'))

In [ ]:
# Multi-Ethnic Study of Atherosclerosis (MESA)
raw = circStudio.io.read_mesa(os.path.join(fpath, 'data', 'test_sample_mesa.csv'))

In [ ]:
# Respironics
raw = circStudio.io.read_rpx(os.path.join(fpath, 'data', 'test_sample_rpx_eng.csv'))

In [ ]:
# Tempatilumi
raw = circStudio.io.read_tal(os.path.join(fpath, 'data', 'test_sample_tal.txt'))

## Retrieving information from `Raw` instances

After converting an original actigraphy file into a `Raw` object, users may access the underlying data and respective metadata. Below are some examples; for a complete description of available attributes and methods, please refer to the `API`.

<div class="alert alert-block alert-info">
    <b>NOTE</b> Only the information required for calculations is imported from the original file. Metadata not used by <code>circStudio</code> is not imported.
</div>

- Activity (`pd.Series`):

In [ ]:
raw.activity

- Interactive activity plot:

In [ ]:
raw.plot(mode='activity', log=False)

- Light (`pd.Series`):

In [ ]:
raw.light

In [ ]:
# Create a new figure and axis
raw.plot(mode='light', log=True)

- Plot of the temperature signal (extracted from `raw.df`):

In [ ]:
raw.plot(ts='TEMPERATURE', log='True')

- First five rows of the `pd.DataFrame`:

In [ ]:
raw.df.head()

- Acquisition frequency:

In [ ]:
raw.frequency

- Duration of the data acquisition period:

In [ ]:
raw.duration()

In the next section, we will explore data masking and resampling using methods provided by the `Raw` class.